# Data Acquisition via API Calls (NewsAPI)

---

## Phase 1: Setup and Security

### Install Dependencies

In [1]:
# Install the necessary library for loading environment variables (like API keys)
!pip install python-dotenv

print("Dependencies installed successfully.")

Dependencies installed successfully.


### Import Modules
Import all necessary Python libraries. Note that `requests` is the primary tool for submitting HTTP requests.

In [2]:
import numpy as np
import pandas as pd
import os
import requests
import json
import dotenv
from datetime import datetime

print("Modules imported.")

Modules imported.


### Load Environment Variables
This loads your `.env` file, which is crucial for securely handling your NewsAPI key without exposing it in the notebook.

In [3]:
# Load the .env file to access environment variables (e.g., your API Key)
dotenv.load_dotenv()

print("Environment variables loaded.")

Environment variables loaded.


### Retrieve API Key
We access the stored API key using os.getenv().

In [4]:
# Get the News API key from the environment variables
newskey = os.getenv('newskey')

# We won't print the key, but we confirm it's loaded (e.g., by checking its length)
if newskey:
    print("News API key successfully retrieved.")
else:
    print("ERROR: News API key not found. Check your .env file.")

News API key successfully retrieved.


---

## Phase 2: Building the API Request Components
An HTTP request requires four main components: URL root, endpoint, headers (for security/metadata), and parameters (the query).

### Build User Agent Header
Many APIs require a User-agent to identify the client application. We use httpbin.org to dynamically grab a standard User-agent string.

In [5]:
# 1. Get a standard User-agent string
r = requests.get('https://httpbin.org/user-agent')
useragent = json.loads(r.text)['user-agent']

# 2. Build the full headers dictionary
# The API key is sent via the 'X-Api-Key' header, a common secure method.
headers = {'User-agent': useragent,
           'X-Api-Key': newskey}

print(f"User-agent: {useragent}")
print("Headers dictionary created, containing the API key.")

User-agent: python-requests/2.32.5
Headers dictionary created, containing the API key.


### Define URL Root and Endpoint
This defines the fixed address for the NewsAPI service we want to use.

In [6]:
# Define the fixed URL parts
root = 'https://newsapi.org'
endpoint = '/v2/everything' # This endpoint searches all articles

print(f"API Base URL: {root + endpoint}")

API Base URL: https://newsapi.org/v2/everything


### Define Query Parameters
Parameters are used to customize the search (e.g., topic, language, date).

In [7]:
# Define the search parameters as a Python dictionary
params = {'q': '"tallest mountain"',  # Topic to search for (using quotes for exact phrase)
         'searchIn': 'content',      # Search within the article content
         'language': 'en',           # Restrict to English articles
         'pageSize': 100}            # Request up to 100 articles

print("Query parameters defined:")
print(params)

Query parameters defined:
{'q': '"tallest mountain"', 'searchIn': 'content', 'language': 'en', 'pageSize': 100}


---

## Phase 3: API Execution and Parsing

### Submit the GET Request
This cell sends the request and checks the response status. A `<Response [200]>` indicates success.

In [8]:
# Combine the components and submit the GET request
r = requests.get(root + endpoint,
                headers = headers,
                params = params)

# Display the response object (it should show <Response [200]>)
print(f"Request submitted. Status: {r}")

Request submitted. Status: <Response [200]>


### Parse the JSON Response
The response (`r.text`) is a single string containing the JSON data. We use `json.loads()` to convert this string into a usable Python dictionary.

In [9]:
# Convert the JSON response string into a Python dictionary
myjson = json.loads(r.text)

print(f"JSON response converted to a Python dictionary (Type: {type(myjson)})")
print("Top-level keys in the response:")
print(list(myjson.keys()))

JSON response converted to a Python dictionary (Type: <class 'dict'>)
Top-level keys in the response:
['status', 'totalResults', 'articles']


### View Raw JSON Data Structure (Optional)
This cell is often left commented out to avoid printing a massive wall of text but serves as a way to inspect the data structure.

In [10]:
# Uncomment this line to inspect the full structure of the JSON dictionary
# print(json.dumps(myjson, indent=4))

### Normalize JSON to DataFrame
The key data is nested under the articles key. Pandas’ json_normalize() function flattens this nested data into a clean, tabular DataFrame.

In [11]:
# Use json_normalize to extract the list of article dictionaries ('articles')
news_df = pd.json_normalize(myjson, record_path = ['articles'])

print(f"DataFrame created with {len(news_df)} articles.")
print("First 5 rows of the DataFrame:")
display(news_df.head())

DataFrame created with 20 articles.
First 5 rows of the DataFrame:


,author,title,description,url,urlToImage,publishedAt,content,source.id,source.name
0,None,Last surviving member of first team to scale E...,Kanchha Sherpa was 19 when he accompanied the ...,https://www.bbc.com/news/articles/ceq0xlnv2x2o,https://ichef.bbci.co.uk/news/1024/branded_new...,2025-10-16T11:23:41Z,"Kanchha Sherpa, the last surviving member of t...",None,BBC News
1,Al Jazeera,India’s Himalayan villages slowly reviving dec...,People return to Himalayan villages each summe...,https://www.aljazeera.com/gallery/2025/10/15/l...,https://www.aljazeera.com/wp-content/uploads/2...,2025-10-15T10:12:21Z,Dozens of dilapidated stone buildings are all ...,al-jazeera-english,Al Jazeera English
2,SATISH SHARMA,Photos show life slowly returning to abandoned...,Dozens of dilapidated stone buildings are what...,https://www.yahoo.com/news/articles/photos-sho...,https://s.yimg.com/ny/api/res/1.2/kw3pDflmNQqc...,2025-10-15T03:44:20Z,"MARTOLI, India (AP) Dozens of dilapidated ston...",None,Yahoo Entertainment
3,Sarah Thwaites,"Brink Traveler Adds New Locations, Hand Tracki...",VR travel app Brink Traveler added new locatio...,https://www.uploadvr.com/brink-traveller-itali...,https://www.uploadvr.com/content/images/size/w...,2025-10-29T12:30:59Z,Brink Traveler's latest update sees the travel...,None,UploadVR
4,Tomas Pueyo,Argentina Could Be a Superpower,Its geography could have made it the United St...,https://unchartedterritories.tomaspueyo.com/p/...,"https://substackcdn.com/image/fetch/$s_!2UQQ!,...",2025-10-21T01:06:55Z,Encuentra este artículo en español debajo del ...,None,Tomaspueyo.com


---

## Phase 4: Data Analysis and Export

### Clean and Prepare for Export
Perform a final step to ensure the data is properly formatted before saving.

In [12]:
# Clean up the publishedAt column (convert to datetime)
news_df['publishedAt'] = pd.to_datetime(news_df['publishedAt'])

# Select a final set of columns for the CSV
final_df = news_df[['publishedAt', 'title', 'description', 'url', 'source.name', 'author']].copy()

print("DataFrame prepared for export.")

DataFrame prepared for export.


### Export to CSV
This is the final Load (L) phase of the data acquisition process.

In [13]:
# Define a filename based on the current date for organization
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_filename = f'news_articles_{timestamp}.csv'

# Export the final clean DataFrame to a CSV file
final_df.to_csv(output_filename, index=False)

print(f"Successfully exported {len(final_df)} articles to {output_filename}")

Successfully exported 20 articles to news_articles_20251115_223435.csv


---

## Function Definition (Optional Consolidation)
This final cell provides the consolidation of all steps into a single, reusable function—demonstrating how a script would execute the entire API call process.

In [15]:
def grab_latest_articles():
    """
    Consolidates all steps: builds request, calls API, parses JSON, and returns a DataFrame.
    """
    
    # Prompt the user
    topic = input("Please enter your topic of interest: ")
    
    # Build our Headers
    newskey = os.getenv('newskey')
    r = requests.get('https://httpbin.org/user-agent')
    useragent = json.loads(r.text)['user-agent']
    headers = {'User-agent': useragent,
               'X-Api-Key': newskey}

    # Build our URL and Parameters
    root = 'https://newsapi.org'
    endpoint = '/v2/everything'
    params = {'q': topic,
              'searchIn': 'content',
              'language': 'en',
              'pageSize': 100}
    
    # Submit our Request
    r = requests.get(root + endpoint,
                headers = headers,
                params = params)
    
    # Create and return the pandas dataframe
    myjson = json.loads(r.text)
    news_df = pd.json_normalize(myjson, record_path = ['articles'])
    
    return news_df

In [16]:
grab_latest_articles()

,author,title,description,url,urlToImage,publishedAt,content,source.id,source.name
0,Justin Carter,Toy Company Funko Has ‘Going Concerns’ About I...,Things haven't been going well for Funko finan...,https://gizmodo.com/toy-company-funko-has-goin...,https://gizmodo.com/app/uploads/2025/11/AlienE...,2025-11-09T17:25:44Z,"For over a decade, Funko Pops have grown into ...",None,Gizmodo.com
1,Sabina Graves,The Best Memes About Disney’s ‘Soarin’ Across ...,The seasonal overlay is a part of Disney Celeb...,https://gizmodo.com/the-best-memes-about-disne...,https://gizmodo.com/app/uploads/2025/10/Soarin...,2025-10-21T10:00:13Z,There’s going to be a patriotic new Disney Par...,None,Gizmodo.com
2,Ece Yildirim,Nvidia CEO Jensen Huang Makes His Case for Chi...,"Huang now joins Trump in his Asia tour, ahead ...",https://gizmodo.com/nvidia-ceo-jensen-huang-ma...,https://gizmodo.com/app/uploads/2025/10/shutte...,2025-10-29T23:00:47Z,Nvidia’s first ever GTC to be hosted in Washin...,None,Gizmodo.com
3,Lucas Ropek,New Trump Administration Energy Rule Would Ena...,"The ""urgent"" regulatory amendment from the Ene...",https://gizmodo.com/ai-large-loads-2000676957,https://gizmodo.com/app/uploads/2025/10/data-c...,2025-10-25T14:00:39Z,Data centers are being built with unprecedente...,None,Gizmodo.com
4,Kim LaCapria,Elon Musk issues ominous warning amid struggli...,Musk has been claiming the United States would...,https://finance.yahoo.com/news/elon-musk-issue...,https://s.yimg.com/ny/api/res/1.2/DAZNtlKbssVd...,2025-11-05T11:50:00Z,Tesla CEO Elon Musk revisited Joe Rogan's podc...,None,Yahoo Entertainment
...,...,...,...,...,...,...,...,...,...
94,Express Web Desk,Trump ends ‘all trade negotiations’ with Canad...,"In a post on social media, Trump accused Ottaw...",https://indianexpress.com/article/world/trump-...,https://images.indianexpress.com/2025/10/Trump...,2025-10-24T03:30:50Z,US President Donald Trump announced on Thursda...,None,The Indian Express
95,Al Jazeera,Japan’s parliament confirms hardliner Takaichi...,Appointment clinched via a last-minute coaliti...,https://www.aljazeera.com/news/2025/10/21/japa...,https://www.aljazeera.com/wp-content/uploads/2...,2025-10-21T07:08:48Z,Japans parliament has elected ultraconservativ...,al-jazeera-english,Al Jazeera English
96,"Brian Domitrovic, Contributor, \n Brian Domitr...",Returning To A Gold Standard Has Been Done Before,"In major episodes of economic history, includi...",https://www.forbes.com/sites/briandomitrovic/2...,https://imageio.forbes.com/specials-images/ima...,2025-10-27T10:12:11Z,"Wells Fargo moving gold, Deadwood, S.D., 1890 ...",None,Forbes
97,Al Jazeera,US aims to raise $20bn ‘facility’ to support A...,"The additional boost, which comes on top of a ...",https://www.aljazeera.com/news/2025/10/15/us-a...,https://www.aljazeera.com/wp-content/uploads/2...,2025-10-15T18:17:11Z,"The head of the United States Treasury, Scott ...",al-jazeera-english,Al Jazeera English
